## CKA to PESQ

A per-utterance correlation between layer-wise CKA and PESQ improvement is dominated by SNR: both quantities track degradation severity. To isolate any predictive relationship internal to a fixed degradation level, we remove the SNR-dependent mean effect by within-group centering before correlating. Encoder and latent layers correlate near zero. Decoder and refinement layers show increasingly negative centered correlations through the first decoder block, which then saturate.

Runtime: under 30 seconds on any device.


In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
while REPO.name != "SE-Probe" and REPO.parent != REPO:
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from se_probe.device import device_info, get_device
from se_probe.plotting import apply_paper_rcparams

apply_paper_rcparams()

DEVICE = get_device()
print(device_info(DEVICE))

import pandas as pd

DATA_DIR = REPO / ("results_df" if (REPO / "results_df").exists() else "results_demo")
print(f"Using data from: {DATA_DIR}")

## Load MUSE table and compute centered residuals

PESQ gain = enhanced_pesq − noisy_pesq, then subtract the per-(layer, SNR) cell mean for both CKA and PESQ-gain. The centered quantities isolate any predictive relationship internal to a fixed degradation level.

In [ ]:
muse_path = DATA_DIR / ("snr/cka_snr_muse.parquet" if (DATA_DIR / "snr").exists() else "cka_snr_muse_demo.parquet")
df = pd.read_parquet(muse_path)
df = df[df["layer"].str.endswith(".norm1")].copy()

block_order = ["encoder_level1", "encoder_level2", "latent", "decoder_level2", "decoder_level1", "mag_refinement"]
NORM1_LAYERS = sorted(df["layer"].unique(),
                      key=lambda L: (next((i for i, b in enumerate(block_order) if b in L), len(block_order)), L))

df["PESQ_gain"] = df["enhanced_pesq"] - df["noisy_pesq"]
for col in ["CKA", "PESQ_gain"]:
    df[col + "_centered"] = df[col] - df.groupby(["layer", "snr"])[col].transform("mean")

print(f"norm1 layers: {len(NORM1_LAYERS)} | rows after filter: {len(df):,}")

## Per-layer correlation

In [ ]:
layer_correlations = []
for layer in NORM1_LAYERS:
    layer_data = df[df["layer"] == layer]
    layer_correlations.append(layer_data["CKA_centered"].corr(layer_data["PESQ_gain_centered"]))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 3))
x = range(len(NORM1_LAYERS))
ax.plot(x, layer_correlations, "o-", markersize=8, linewidth=2)
ax.set_xlabel("Block")
ax.set_ylabel(r"$\rho$", rotation=90)
ax.set_title("Correlation between Centered CKA and PESQ Gain by Layer")

num_blocks = 6
block_size = 4
tick_positions = [i * block_size + block_size / 2 - 1.5 for i in range(num_blocks)]
tick_labels = ["Enc-L1", "Enc-L2", "Latent", "Dec-L2", "Dec-L1", "Refinement"]
ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels, rotation=30, ha="center")

for i in range(block_size, len(NORM1_LAYERS), block_size):
    ax.axvline(x=i, color="k", linestyle="--", linewidth=1.2)

ax.grid(True, alpha=0.3)
ax.axhline(y=0, color="red", linestyle="--", alpha=0.7, label="Zero correlation")
ax.legend()
plt.tight_layout()